# OLS, PLS, PCR, ElasticNet, RandomForest
This notebook contains the code for the OLS, PLS, PCR, ElasticNet and RandomForest models. 

## Imports

In [1]:
import pandas as pd
import wrds 
import numpy as np
from sklearn.linear_model import ElasticNet
import tidyfinance as tf
import sqlite3
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

tf.set_wrds_credentials()

Invalid choice. Please start again and enter 'project' or 'home'.


## Settings

In [ ]:
DATA_ALREADY_PROCESSED = False
LOCATION_OF_CHARACTERISTICS = './datashare.csv'
LOCATION_OF_FINAL_DATA = 'final_data.csv'
WRITE_TO_SUB_FILES = False

## Data preparation

In [ ]:
if DATA_ALREADY_PROCESSED:
    df = pd.read_csv('final_data.csv', low_memory=False)
else:
    characteristics = pd.read_csv(r"./datashare.csv", low_memory=False)    
    characteristics.rename(columns={'DATE': 'month'}, inplace=True)    
    characteristics['month'] = pd.to_datetime(characteristics['month'], format='%Y%m%d')
    characteristics['month'] = characteristics['month'].dt.to_period('M').dt.to_timestamp()    
    cols_to_prefix = {col: f"characteristic_{col}" for col in characteristics.columns 
                    if col not in ['permno', 'month', 'sic2']}
    characteristics.rename(columns=cols_to_prefix, inplace=True)    
    characteristics.dropna(subset=['sic2'], inplace=True)    
    characteristics['sic2'] = characteristics['sic2'].astype('category')  
    
    if WRITE_TO_SUB_FILES:
        characteristics.to_csv(r"./characteristics.csv", index=False) 
    
    def rank_transform(x):
        rank_x = x.rank(method='average')                
        non_missing = x.dropna()
        max_rank = non_missing.shape[0]
        min_rank = 1
                
        if max_rank == 0:
            return pd.Series([pd.NA] * len(x), index=x.index)
        else:        
            return 2 * ((rank_x - min_rank) / (max_rank - min_rank) - 0.5)
    
    cols = [col for col in characteristics.columns if "characteristic" in col]
    characteristics[cols] = characteristics.groupby('month')[cols].transform(rank_transform)
    
    if WRITE_TO_SUB_FILES:
        characteristics.to_csv(r"./characteristics_ranked.csv", index=False)
    
    cols = [col for col in characteristics.columns if "characteristic" in col]

    characteristics[cols] = characteristics.groupby('month')[cols].transform(lambda x: x.fillna(x.median(skipna=True)))
    characteristics[cols] = characteristics[cols].fillna(0)
    
    db = sqlite3.connect('data.db')
    
    crsp_monthly = tf.download_data(
    domain="wrds",
    dataset="crsp_monthly",
    start_date='1957-01-01',
    end_date='2016-12-31'
    )

    crsp_monthly = (crsp_monthly
        .dropna(subset=["ret_excess", "mktcap", "mktcap_lag"])
    )

    (crsp_monthly
    .to_sql(name="crsp_monthly", 
            con=db, 
            if_exists="replace",
            index=False)
    )
    
    df_macro_pred = tf.download_data(
    domain="macro_predictors",
    dataset="monthly",
    start_date='1957-01-01', 
    end_date='2016-12-31'
    )

    df_macro_pred.to_sql(
    name="macro_predictors",
    con=db, 
    if_exists="replace",
    index=False
    )
    
    crsp_monthly = pd.read_sql_query(
    "SELECT date, permno, mktcap_lag, ret_excess FROM crsp_monthly", 
    db
)
crsp_monthly['month'] = pd.to_datetime(crsp_monthly['date'])
macro_predictors = pd.read_sql_query(
    "SELECT date, dp, ep, bm, ntis, tbl, tms, dfy, svar FROM macro_predictors", 
    db
)
macro_predictors['month'] = pd.to_datetime(macro_predictors['date'])
macro_predictors = macro_predictors.rename(
    columns=lambda x: f"macro_{x}" if x != "month" else x
)
macro_shifted = macro_predictors.copy()
macro_shifted['month'] = macro_shifted['month'] + pd.offsets.MonthBegin(1)
macro_predictors_final = pd.merge(
    macro_predictors[['month']],
    macro_shifted, 
    on='month', 
    how='left'
)
characteristics['month'] = pd.to_datetime(characteristics['month'])
characteristics
df = characteristics.merge(crsp_monthly, on=['month', 'permno'], how='inner')
df = df.merge(macro_predictors_final, on='month', how='inner')
df.sort_values(['month', 'permno'], inplace=True)
df = df.drop(columns=['macro_date'], axis=1)
df['macro_intercept'] = 1
selected_cols = ['permno', 'month', 'ret_excess', 'mktcap_lag', 'sic2']
selected_cols += [col for col in df.columns if 'macro' in col or 'characteristic' in col]
df = df[selected_cols]

market_cap_threshold = df['mktcap_lag'].quantile(0.2)
df = df[df['mktcap_lag'] >= market_cap_threshold]

## Plot OOS years

In [ ]:
validation_length = 12

oos_years = range(1987, 2022)  # 1987 to 2021 inclusive
estimation_periods = pd.DataFrame({
    'oos_year': list(oos_years)
})
estimation_periods['validation_end'] = estimation_periods['oos_year'] - 1
estimation_periods['validation_start'] = estimation_periods['oos_year'] - validation_length
estimation_periods['training_start'] = 1957
estimation_periods['training_end'] = estimation_periods['validation_start'] - 1

cols = [col for col in estimation_periods.columns if 'training' in col] + ['validation_start', 'validation_end', 'oos_year']
estimation_periods = estimation_periods[cols]

rows = []
for _, row in estimation_periods.iterrows():
    for year in range(1957, 2022):
        classification = None    
        if row['training_start'] <= year <= row['training_end']:
            classification = "Training"
        elif row['validation_start'] <= year <= row['validation_end']:
            classification = "Validation"
        elif year == row['oos_year']:
            classification = "OOS"
                            
        if classification is not None:
            rows.append({
                'year': year,
                'oos_year': row['oos_year'],
                'classification': classification
            })
visualization_data = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(10, 6))

for cls in visualization_data['classification'].unique():
    subset = visualization_data[visualization_data['classification'] == cls]
    ax.scatter(subset['year'], subset['oos_year'], label=cls, s=20)  # s controls marker size

ax.set_title("Data classification timeline")
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_yticks([])
ax.tick_params(axis='y', which='both', length=0)

ax.legend(title="")

plt.tight_layout()
plt.show()

## OLS

In [ ]:
exclude_cols = [col for col in ['permno', 'month', 'year', 'mktcap_lag', 'ret_excess'] if col in df.columns]

oos_years = range(2000, 2017)  # 2000 to 2016 inclusive
df['year'] = pd.to_datetime(df['month']).dt.year

def calculate_oos_r2(y_true, y_pred, y_train_mean):
    ss_res = np.sum((y_true - y_pred) ** 2)  
    ss_tot = np.sum((y_true - y_train_mean) ** 2)  
    if ss_tot == 0:
        return np.nan
    return 1 - ss_res / ss_tot

def calculate_naive_oos_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)  
    ss_tot = np.sum((y_true) ** 2)
    if ss_tot == 0: 
        return np.nan
    return 1 - ss_res / ss_tot

oos_r2_results = []
for oos_year in oos_years:
    train_data = df[(df['year'] >= 1980) & (df['year'] < oos_year)]    
    oos_data = df[df['year'] == oos_year]
    
    if train_data.empty or oos_data.empty:
        print(f"Skipping OOS year {oos_year}: Empty training or OOS data.")
        oos_r2_results.append({'oos_year': oos_year, 'oos_r2': np.nan})
        continue

    y_train = train_data['ret_excess']
    X_train = train_data.drop(columns=exclude_cols)
    X_train = X_train.fillna(0)
    X_train = sm.add_constant(X_train)

    y_oos = oos_data['ret_excess']
    X_oos = oos_data.drop(columns=exclude_cols)
    X_oos = X_oos.fillna(0)
    X_oos = sm.add_constant(X_oos)

    if X_train.columns.tolist() != X_oos.columns.tolist():
        print(f"Warning: Column mismatch in OOS year {oos_year}. Aligning columns.")
        common_cols = X_train.columns.intersection(X_oos.columns)
        X_train = X_train[common_cols]
        X_oos = X_oos[common_cols]
    
    try:
        model = sm.OLS(y_train, X_train)
        results = model.fit()

        y_pred = results.predict(X_oos)
        y_train_mean = y_train.mean()

        oos_r2 = calculate_oos_r2(y_oos, y_pred, y_train_mean)
        naive_r2 = calculate_naive_oos_r2(y_oos, y_pred)
        oos_r2_results.append({'oos_year': oos_year, 'oos_r2': oos_r2})
        print(f"OOS Year {oos_year}: OOS R-squared = {oos_r2:.6f}")
        
    except Exception as e:
        print(f"Error in OOS year {oos_year}: {str(e)}")
        oos_r2_results.append({'oos_year': oos_year, 'oos_r2': np.nan})

oos_r2_df = pd.DataFrame(oos_r2_results)
print("\nOut-of-Sample R-squared Results:")
print(oos_r2_df)

oos_r2_df.to_csv('oos_r2_results.csv', index=False)
with open('oos_r2_summary.txt', 'w') as f:
    f.write("Out-of-Sample R-squared Results:\n")
    f.write(oos_r2_df.to_string(index=False))
    f.write(f"\n\nAverage OOS R-squared: {oos_r2_df['oos_r2'].mean():.6f}")
    f.write(f"\nMedian OOS R-squared: {oos_r2_df['oos_r2'].median():.6f}")
    

## PLS

In [ ]:
base_exclude = ['permno','month','ret_excess','mktcap_lag','macro_intercept']
portfolio_exclude = ['portfolio_score','portfolio_rank','ridge_portfolio_score',
                     'lasso_portfolio_score','ridge_portfolio_rank','lasso_portfolio_rank']
exclude_cols = [c for c in base_exclude + portfolio_exclude if c in df.columns]

results = []
for oos in oos_years:
    train = df[(df['year'] >= 1980) & (df['year'] < oos)]
    test  = df[df['year'] == oos]
    if train.empty or test.empty:
        results.append({'oos_year': oos, 'oos_r2': np.nan, 'naive_r2': np.nan})
        continue
    X_train = train.drop(columns=exclude_cols)
    y_train = train['ret_excess']
    X_test  = test.drop(columns=exclude_cols)
    y_test  = test['ret_excess']
    mu = X_train.mean()
    sigma = X_train.std().replace(0, 1)
    X_train_std = (X_train - mu) / sigma
    X_test_std  = (X_test  - mu) / sigma
    pls = PLSRegression(n_components=10)
    pls.fit(X_train_std, y_train)
    y_pred = pls.predict(X_test_std).ravel()
    oos_r2   = calculate_oos_r2(y_test.values, y_pred, y_train.mean())
    naive_r2 = calculate_naive_oos_r2(y_test.values, y_pred)
    results.append({'oos_year': oos, 'oos_r2': oos_r2, 'naive_r2': naive_r2})
    print(f"OOS {oos}: PLS OOS R² = {oos_r2:.4f}, Naive R² = {naive_r2:.4f}")

oos_pls_df = pd.DataFrame(results)
print("\nPLS Out-of-Sample R²:")
print(oos_pls_df)
print(f"\nAvg OOS R²: {oos_pls_df['oos_r2'].mean():.4f}")
print(f"Med OOS R²: {oos_pls_df['oos_r2'].median():.4f}")

oos_pls_df.to_csv('pls_oos_r2.csv', index=False)

## PCR

In [ ]:
base_exclude = ['permno', 'month', 'ret_excess', 'mktcap_lag', 'macro_intercept']
portfolio_exclude = ['portfolio_score', 'portfolio_rank', 'ridge_portfolio_score',
                     'lasso_portfolio_score', 'ridge_portfolio_rank', 'lasso_portfolio_rank']
exclude_cols = [c for c in base_exclude + portfolio_exclude if c in df.columns]

results = []
for oos in oos_years:
    train = df[(df['year'] >= 1980) & (df['year'] < oos)]
    test = df[df['year'] == oos]
    if train.empty or test.empty:
        results.append({'oos_year': oos, 'oos_r2': np.nan, 'naive_r2': np.nan})
        continue
    X_train = train.drop(columns=exclude_cols).replace('NaN', np.nan).dropna()
    y_train = train.loc[X_train.index, 'ret_excess']
    X_test = test.drop(columns=exclude_cols).replace('NaN', np.nan).dropna()
    y_test = test.loc[X_test.index, 'ret_excess']
    mu = X_train.mean()
    sigma = X_train.std().replace(0, 1)
    X_train_std = (X_train - mu) / sigma
    X_test_std = (X_test - mu) / sigma
    pca = PCA(n_components=10)
    X_train_pca = pca.fit_transform(X_train_std)
    X_test_pca = pca.transform(X_test_std)
    reg = LinearRegression()
    reg.fit(X_train_pca, y_train)
    y_pred = reg.predict(X_test_pca)
    oos_r2 = calculate_oos_r2(y_test.values, y_pred, y_train.mean())
    naive_r2 = calculate_naive_oos_r2(y_test.values, y_pred)
    results.append({'oos_year': oos, 'oos_r2': oos_r2, 'naive_r2': naive_r2})
    print(f"OOS {oos}: PCA OOS R² = {oos_r2:.4f}, Naive R² = {naive_r2:.4f}")

oos_pca_df = pd.DataFrame(results)
print("\nPCA Out-of-Sample R²:")
print(oos_pca_df)
print(f"\nAvg OOS R²: {oos_pca_df['oos_r2'].mean():.4f}")
print(f"Med OOS R²: {oos_pca_df['oos_r2'].median():.4f}")

oos_pca_df.to_csv('pca_oos_r2.csv', index=False)

## Elastic Net

In [ ]:
base_exclude = ['permno','month','ret_excess','mktcap_lag','macro_intercept']
portfolio_exclude = ['portfolio_score','portfolio_rank','ridge_portfolio_score',
                     'lasso_portfolio_score','ridge_portfolio_rank','lasso_portfolio_rank']
exclude_cols = [c for c in base_exclude + portfolio_exclude if c in df.columns]

results = []
for oos in oos_years:
    train = df[(df['year'] >= 1980) & (df['year'] < oos)]
    test  = df[df['year'] == oos]
    if train.empty or test.empty:
        results.append({'oos_year': oos, 'oos_r2': np.nan, 'naive_r2': np.nan})
        continue

    X_train = train.drop(columns=exclude_cols)
    y_train = train['ret_excess']
    X_test  = test.drop(columns=exclude_cols)
    y_test  = test['ret_excess']

    mu = X_train.mean()
    sigma = X_train.std().replace(0, 1)
    X_train_std = (X_train - mu) / sigma
    X_test_std  = (X_test  - mu) / sigma

    en = ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=42)
    en.fit(X_train_std, y_train)
    y_pred = en.predict(X_test_std)

    oos_r2   = calculate_oos_r2(y_test.values, y_pred, y_train.mean())
    naive_r2 = calculate_naive_oos_r2(y_test.values, y_pred)

    results.append({
        'oos_year': oos,
        'oos_r2':   oos_r2,
        'naive_r2': naive_r2
    })
    print(f"OOS {oos}: Elastic Net OOS R² = {oos_r2:.4f}, Naive R² = {naive_r2:.4f}")

oos_en_df = pd.DataFrame(results)
print("\nElastic Net Out-of-Sample R²:")
print(oos_en_df)
print(f"\nAvg OOS R²: {oos_en_df['oos_r2'].mean():.4f}")
print(f"Med OOS R²: {oos_en_df['oos_r2'].median():.4f}")

oos_en_df.to_csv('elastic_net_oos_r2.csv', index=False)

## Random forest

In [ ]:
base_exclude = ['permno','month','ret_excess','mktcap_lag','macro_intercept']
portfolio_exclude = [
    'portfolio_score','portfolio_rank','ridge_portfolio_score','lasso_portfolio_score',
    'ridge_portfolio_rank','lasso_portfolio_rank','elastic_net_portfolio_score',
    'elastic_net_portfolio_rank','pca_portfolio_score','pca_portfolio_rank',
    'pls_portfolio_score','pls_portfolio_rank','gbr_portfolio_score','gbr_portfolio_rank'
]
exclude_cols = [c for c in base_exclude + portfolio_exclude if c in df.columns]

results = []
for oos in oos_years:
    train = df[(df['year'] >= 1980) & (df['year'] < oos)]
    test  = df[df['year'] == oos]
    if train.empty or test.empty:
        results.append({'oos_year': oos, 'oos_r2': np.nan, 'naive_r2': np.nan})
        continue
    X_train = train.drop(columns=exclude_cols)
    y_train = train['ret_excess']
    X_test  = test.drop(columns=exclude_cols)
    y_test  = test['ret_excess']
    rf = RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    oos_r2   = calculate_oos_r2(y_test.values, y_pred, y_train.mean())
    naive_r2 = calculate_naive_oos_r2(y_test.values, y_pred)
    results.append({'oos_year': oos, 'oos_r2': oos_r2, 'naive_r2': naive_r2})
    print(f"OOS {oos}: RF OOS R² = {oos_r2:.4f}, Naive R² = {naive_r2:.4f}")

oos_rf_df = pd.DataFrame(results)
print("\nRF Out-of-Sample R²:")
print(oos_rf_df)
print(f"\nAvg OOS R²: {oos_rf_df['oos_r2'].mean():.4f}")
print(f"Med OOS R²: {oos_rf_df['oos_r2'].median():.4f}")

oos_rf_df.to_csv('rf_oos_r2.csv', index=False)